# Projeto Capstone: Caracterização de Pêndulo Físico

**Disciplina:** FSC1189 - Algoritmo e Programação

**Objetivo:** Integrar todos os conceitos do curso em um projeto real de análise de dados experimentais.

**Tema:** Caracterização de um pêndulo físico: da medição à publicação.

Este projeto demonstra:
- Coleta e tratamento de dados
- Análise estatística com incertezas
- Modelagem numérica (Euler vs Runge-Kutta)
- Visualização científica
- Comunicação de resultados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Configurar matplotlib para qualidade de publicação
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

## Bloco 1: Campanha de Medição

Simular uma campanha de medições do período de um pêndulo para 9 comprimentos diferentes.

In [ ]:
class CampanhaPendulo:
    """Simula uma campanha de medições de período de pêndulo."""
    
    def __init__(self, g_real=9.81, seed=42):
        np.random.seed(seed)
        self.g_real = g_real
    
    def periodo_teorico(self, L):
        """T = 2π√(L/g) para pequenos ângulos."""
        return 2 * np.pi * np.sqrt(L / self.g_real)
    
    def gerar_campanha(self, comprimentos, num_medidas=15):
        """Gera dados de uma campanha de medições."""
        dados = []
        sigma_percent = 0.015  # 1.5% de incerteza
        
        for L in comprimentos:
            T_teorico = self.periodo_teorico(L)
            sigma = sigma_percent * T_teorico
            
            # Gerar medidas
            for i in range(num_medidas):
                T_med = np.random.normal(T_teorico, sigma)
                # Adicionar 1-2 outliers por comprimento
                if np.random.rand() < 0.07:
                    T_med *= (1 + np.random.uniform(0.08, 0.12) * np.sign(np.random.randn()))
                
                dados.append({
                    'L (m)': L,
                    'T (s)': T_med,
                    'T_teor (s)': T_teorico,
                    'Medida': i + 1
                })
        
        return pd.DataFrame(dados)

# Executar campanha
campanha = CampanhaPendulo(g_real=9.81)
comprimentos = np.linspace(0.25, 2.0, 9)  # 0.25m a 2.0m
df = campanha.gerar_campanha(comprimentos, num_medidas=15)

print("=== CAMPANHA DE MEDIÇÃO ===")
print(f"Comprimentos: {len(comprimentos)}")
print(f"Medidas por comprimento: 15")
print(f"Total de pontos: {len(df)}")
print()
print("Intervalo de comprimentos:", f"{comprimentos.min():.2f}m a {comprimentos.max():.2f}m")
print("Intervalo de períodos:", f"{df['T (s)'].min():.3f}s a {df['T (s)'].max():.3f}s")

## Bloco 2: Análise Estatística com Propagação de Incerteza

In [ ]:
# Remover outliers
def remover_outliers(dados, col, threshold=2.5):
    z_scores = np.abs(stats.zscore(dados[col]))
    return dados[z_scores < threshold].copy()

df_clean = remover_outliers(df, 'T (s)', threshold=2.5)

print(f"Outliers removidos: {len(df) - len(df_clean)}")

# Calcular estatísticas por comprimento
stats_por_L = []

for L in comprimentos:
    mask = df_clean['L (m)'] == L
    T_medidas = df_clean[mask]['T (s)'].values
    
    T_meio = T_medidas.mean()
    sigma_T = T_medidas.std() / np.sqrt(len(T_medidas))  # erro da média
    
    # Estimar g a partir de T = 2π√(L/g) => g = 4π²L/T²
    g_estimado = 4 * np.pi**2 * L / T_meio**2
    
    # Propagação de incerteza: σ_g = |dg/dT| * σ_T = -2g/T * σ_T
    sigma_g = 2 * g_estimado / T_meio * sigma_T
    
    stats_por_L.append({
    'L (m)': L,
        'T_meio (s)': T_meio,
        'sigma_T (s)': sigma_T,
        'g (m/s²)': g_estimado,
        'sigma_g (m/s²)': sigma_g,
        'N': len(T_medidas)
    })

df_stats = pd.DataFrame(stats_por_L)

print("\n=== ESTIMATIVAS DE g POR COMPRIMENTO ===")
print(df_stats[['L (m)', 'T_meio (s)', 'g (m/s²)', 'sigma_g (m/s²)']].to_string(index=False))

# g global com erro
g_global = df_stats['g (m/s²)'].mean()
sigma_g_global = np.sqrt(np.sum(df_stats['sigma_g (m/s²)']**2)) / len(df_stats)

print(f"\nValor global de g: {g_global:.4f} ± {sigma_g_global:.4f} m/s²")
print(f"Erro percentual: {abs(g_global - 9.81) / 9.81 * 100:.2f}%")

## Bloco 3: Comparação Euler vs Runge-Kutta para Pêndulo Não-Linear

Demonstrar quando a aproximação linear quebra.

In [ ]:
def pendulo_nao_linear(theta, t, L, g):
    """Pêndulo não-linear: d²θ/dt² = -(g/L)sin(θ)"""
    dtheta_dt = theta[1]
    d2theta_dt2 = -(g / L) * np.sin(theta[0])
    return [dtheta_dt, d2theta_dt2]

def pendulo_linear(theta, t, L, g):
    """Pêndulo linear: d²θ/dt² = -(g/L)θ (aproximação para pequenos ângulos)"""
    dtheta_dt = theta[1]
    d2theta_dt2 = -(g / L) * theta[0]
    return [dtheta_dt, d2theta_dt2]

# Parâmetros
L = 1.0  # 1 metro
g = 9.81
angulos = [5, 30, 60, 90]  # graus
t = np.linspace(0, 3, 300)  # 3 segundos

resultados = {}

for theta0_graus in angulos:
    theta0_rad = np.radians(theta0_graus)
    cond_ini = [theta0_rad, 0]
    
    # Resolver
    sol_nao_linear = odeint(pendulo_nao_linear, cond_ini, t, args=(L, g))
    sol_linear = odeint(pendulo_linear, cond_ini, t, args=(L, g))
    
    resultados[theta0_graus] = {
        'nao_linear': sol_nao_linear[:, 0] * 180/np.pi,  # converter para graus
        'linear': sol_linear[:, 0] * 180/np.pi,
        'periodo_real': sol_nao_linear[1:, 0] * 180/np.pi,
        'periodo_aprox': sol_linear[1:, 0] * 180/np.pi
    }

# Visualização
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for idx, theta0 in enumerate(angulos):
    ax = axes[idx // 2, idx % 2]
    
    ax.plot(t, resultados[theta0]['nao_linear'], 'b-', linewidth=2.5, label='Não-linear')
    ax.plot(t, resultados[theta0]['linear'], 'r--', linewidth=2, alpha=0.7, label='Linear (aprox.)')
    ax.fill_between(t, resultados[theta0]['nao_linear'], resultados[theta0]['linear'], 
                    alpha=0.2, color='yellow')
    
    ax.set_xlabel('Tempo (s)', fontsize=10)
    ax.set_ylabel('Ângulo (graus)', fontsize=10)
    ax.set_title(f'Pêndulo: θ₀ = {theta0}°', fontsize=11, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-theta0 - 5, theta0 + 5)

plt.suptitle('Validação da Aproximação Linear: Pêndulo para Diferentes Amplitudes', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=== VALIDAÇÃO DA APROXIMAÇÃO LINEAR ===")
for theta0 in angulos:
    erro_max = np.max(np.abs(resultados[theta0]['nao_linear'] - resultados[theta0]['linear']))
    print(f"θ₀ = {theta0:2}°: erro máximo = {erro_max:.3f}°")

## Bloco 4: Figura de Publicação (Grid 2×3)

Criar uma figura profissional pronta para publicação ou apresentação.

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.35)

# Painel 1: Regressão Linear T² vs L
ax1 = fig.add_subplot(gs[0, 0])
L_stats = df_stats['L (m)'].values
T2_stats = df_stats['T_meio (s)'].values ** 2
sigma_T2 = 2 * df_stats['T_meio (s)'].values * df_stats['sigma_T (s)'].values

# Regressão linear ponderada
weights = 1 / sigma_T2**2
coeff = np.polyfit(L_stats, T2_stats, 1, w=weights)
p_fit = np.poly1d(coeff)
L_fit = np.linspace(L_stats.min(), L_stats.max(), 100)
T2_fit = p_fit(L_fit)

ax1.errorbar(L_stats, T2_stats, yerr=sigma_T2, fmt='bo', markersize=8, capsize=5, capthick=2)
ax1.plot(L_fit, T2_fit, 'r-', linewidth=2.5, label=f'Ajuste: T² = {coeff[0]:.4f}L + {coeff[1]:.4f}')
ax1.set_xlabel('Comprimento L (m)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Período² T² (s²)', fontsize=11, fontweight='bold')
ax1.set_title('(a) Regressão Linear', fontsize=11, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Painel 2: g por ponto com barras de erro
ax2 = fig.add_subplot(gs[0, 1])
ax2.errorbar(df_stats['L (m)'], df_stats['g (m/s²)'], yerr=df_stats['sigma_g (m/s²)'],
            fmt='gs', markersize=8, capsize=5, capthick=2, alpha=0.7, elinewidth=2)
ax2.axhline(9.81, color='k', linestyle='--', linewidth=2, label='g teórico = 9.81 m/s²')
ax2.axhline(g_global, color='r', linestyle='-', linewidth=2, alpha=0.7, label=f'g obtido = {g_global:.3f} m/s²')
ax2.fill_between(df_stats['L (m)'], g_global - sigma_g_global, g_global + sigma_g_global,
                 alpha=0.2, color='red')
ax2.set_xlabel('Comprimento L (m)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Aceleração g (m/s²)', fontsize=11, fontweight='bold')
ax2.set_title('(b) Estimativa de g', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(9.6, 10.1)

# Painel 3: Resíduos
ax3 = fig.add_subplot(gs[0, 2])
residuos = T2_stats - p_fit(L_stats)
ax3.bar(L_stats, residuos, width=0.08, color='purple', alpha=0.7, edgecolor='black')
ax3.axhline(0, color='k', linestyle='-', linewidth=1)
ax3.set_xlabel('Comprimento L (m)', fontsize=11, fontweight='bold')
ax3.set_ylabel('Resíduos T²', fontsize=11, fontweight='bold')
ax3.set_title('(c) Resíduos do Ajuste', fontsize=11, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# Painel 4: Histograma dos valores de g
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(df_stats['g (m/s²)'], bins=5, color='orange', alpha=0.7, edgecolor='black')
ax4.axvline(g_global, color='r', linestyle='--', linewidth=2.5, label=f'Média: {g_global:.3f}')
ax4.axvline(9.81, color='k', linestyle=':', linewidth=2, label='Teórico: 9.81')
ax4.set_xlabel('g (m/s²)', fontsize=11, fontweight='bold')
ax4.set_ylabel('Frequência', fontsize=11, fontweight='bold')
ax4.set_title('(d) Distribuição de g', fontsize=11, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3, axis='y')

# Painel 5: Retrato de fase (pêndulo a 30°)
ax5 = fig.add_subplot(gs[1, 1])
theta0 = np.radians(30)
cond_ini = [theta0, 0]
sol = odeint(pendulo_nao_linear, cond_ini, t, args=(L, g))
theta = sol[:, 0]
omega = sol[:, 1]

ax5.plot(theta * 180/np.pi, omega, 'b-', linewidth=2.5)
ax5.scatter(theta[0] * 180/np.pi, omega[0], s=100, c='red', marker='o', zorder=5, label='Inicial')
ax5.scatter(0, 0, s=100, c='black', marker='x', linewidth=2, zorder=5, label='Equilíbrio')
ax5.set_xlabel('Ângulo θ (graus)', fontsize=11, fontweight='bold')
ax5.set_ylabel('Velocidade ω (rad/s)', fontsize=11, fontweight='bold')
ax5.set_title('(e) Retrato de Fase (30°)', fontsize=11, fontweight='bold')
ax5.legend(fontsize=10)
ax5.grid(True, alpha=0.3)

# Painel 6: Resumo textual
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')

resumo_texto = f"""RESUMO DOS RESULTADOS

━━━━━━━━━━━━━━━━━━━━━━━━━━━
Medições Realizadas
━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Comprimentos: {len(comprimentos)}
• Medidas/comprimento: 15
• Total: {len(df)} pontos
• Outliers removidos: {len(df) - len(df_clean)}

━━━━━━━━━━━━━━━━━━━━━━━━━━━
Resultado Principal
━━━━━━━━━━━━━━━━━━━━━━━━━━━
g = {g_global:.4f} ± {sigma_g_global:.4f} m/s²

Valor teórico: 9.8100 m/s²
Erro: {abs(g_global - 9.81):.4f} m/s²
Erro %: {abs(g_global - 9.81) / 9.81 * 100:.2f}%

━━━━━━━━━━━━━━━━━━━━━━━━━━━
Qualidade do Ajuste
━━━━━━━━━━━━━━━━━━━━━━━━━━━
R² = {1 - np.sum(residuos**2) / np.sum((T2_stats - T2_stats.mean())**2):.4f}
"""

ax6.text(0.05, 0.95, resumo_texto, transform=ax6.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Projeto Capstone: Caracterização de Pêndulo Físico', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('capstone_resultado.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigura salva em 'capstone_resultado.pdf'")

## Bloco 5: Checklist de Competências

In [ ]:
checklist = {
    "Capítulo 1 - Introdução a Algoritmos": [
        "Utilizar estruturas de dados (arrays, DataFrames)",
        "Implementar funções modulares para cálculos"
    ],
    "Capítulo 2 - Lógica de Programação": [
        "Usar condicionais para validar dados",
        "Implementar controles de fluxo em análises"
    ],
    "Capítulo 3 - Variáveis e Tipos": [
        "Manipular arrays NumPy com eficiência",
        "Trabalhar com DataFrames Pandas"
    ],
    "Capítulo 4-6 - Controle e Repetição": [
        "Usar loops para processar múltiplas medições",
        "Aplicar filtros e remoção de outliers"
    ],
    "Capítulo 7 - Funções": [
        "Criar funções reutilizáveis para análises",
        "Organizar código em módulos e classes"
    ],
    "Capítulo 8 - Métodos Numéricos": [
        "Implementar integração numérica (odeint)",
        "Comparar métodos: Euler vs Runge-Kutta"
    ],
    "Capítulo 9 - Tratamento de Dados": [
        "Remover outliers estatisticamente",
        "Calcular incertezas com propagação de erros",
        "Realizar regressão linear ponderada"
    ],
    "Capítulo 10 - Projetos Integrados": [
        "Conduzir uma campanha de medições simulada",
        "Validar modelos lineares vs não-lineares",
        "Produzir figura científica de qualidade"
    ]
}

print("\n" + "="*60)
print("CHECKLIST DE COMPETÊNCIAS - PROJETO CAPSTONE")
print("="*60)

total_itens = 0
for capitulo, itens in checklist.items():
    print(f"\n✓ {capitulo}")
    for item in itens:
        print(f"  ☑ {item}")
        total_itens += 1

print(f"\n{'='*60}")
print(f"Total de competências desenvolvidas: {total_itens}")
print(f"{'='*60}")